# NOTEBOOK : 04_feature_engineering.ipynb
# ROLE     : Compute strictly leak-free chronological match features and export latest team states.
# INPUT    : ../../data/processed/clean_matches.csv
# OUTPUT   : ../../data/processed/features.csv, ../../data/processed/team_latest_state.csv

In [1]:
from pathlib import Path
from collections import defaultdict, deque
import numpy as np
import pandas as pd

INPUT_FILE = Path("../../data/processed/clean_matches.csv")
FEATURES_FILE = Path("../../data/processed/features.csv")
LATEST_STATE_FILE = Path("../../data/processed/team_latest_state.csv")

TARGET_MAP = {"A": 0, "D": 1, "H": 2}
MAX_REST_DAYS = 14

df = pd.read_csv(INPUT_FILE)
df["Match_Date"] = pd.to_datetime(df["Match_Date"])
df = df.sort_values(["Match_Date", "Home_Team"]).reset_index(drop=True)

print(f"Loaded {len(df)} matches from {df['Match_Date'].min().date()} to {df['Match_Date'].max().date()}")

Loaded 2350 matches from 2020-09-12 to 2026-10-19


In [2]:
def compute_window_stats(history_list, window, prior_ppg, prior_gf, prior_ga, prior_wr):
    """Compute rolling stats over the last `window` matches; fall back to leak-free priors if empty."""
    recent = list(history_list)[-window:]
    if not recent:
        return {
            "ppg": prior_ppg,
            "gf": prior_gf,
            "ga": prior_ga,
            "gd": prior_gf - prior_ga,
            "wr": prior_wr,
        }
    n = len(recent)
    pts = sum(m["pts"] for m in recent)
    gf = sum(m["gf"] for m in recent)
    ga = sum(m["ga"] for m in recent)
    wins = sum(1 for m in recent if m["pts"] == 3)
    return {
        "ppg": pts / n,
        "gf": gf / n,
        "ga": ga / n,
        "gd": (gf - ga) / n,
        "wr": wins / n,
    }


def compute_h2h_stats(h2h_list):
    """Compute H2H stats from the perspective of the current Home Team vs Away Team."""
    if not h2h_list:
        return {"h2h_home_wr": 0.43, "h2h_draw_r": 0.24, "h2h_away_wr": 0.33, "h2h_gd": 0.0}
    n = len(h2h_list)
    h_wins = sum(1 for m in h2h_list if m["gd"] > 0)
    draws = sum(1 for m in h2h_list if m["gd"] == 0)
    a_wins = sum(1 for m in h2h_list if m["gd"] < 0)
    avg_gd = sum(m["gd"] for m in h2h_list) / n
    return {
        "h2h_home_wr": h_wins / n,
        "h2h_draw_r": draws / n,
        "h2h_away_wr": a_wins / n,
        "h2h_gd": avg_gd,
    }

In [3]:
# State trackers (updated ONLY AFTER computing features for the current match)
team_all_history = defaultdict(lambda: deque(maxlen=10))
team_home_history = defaultdict(lambda: deque(maxlen=5))
team_away_history = defaultdict(lambda: deque(maxlen=5))
team_last_date = {}
team_season_matches = defaultdict(int)  # keyed by (season, team)
h2h_history = defaultdict(list)         # keyed by (team_a, team_b) -> list of {"gd": team_a_goals - team_b_goals}

# Expanding league-average priors (updated strictly after each match)
league_pts_sum = 0.0
league_gf_sum = 0.0
league_win_sum = 0.0
league_team_games = 0

feature_rows = []

for _, row in df.iterrows():
    season = row["season"]
    m_date = row["Match_Date"]
    home = row["Home_Team"]
    away = row["Away_Team"]
    ftr = row["Full_Time_Result"]
    hg = int(row["Full_Time_Home_Goals"])
    ag = int(row["Full_Time_Away_Goals"])

    # 1. Leak-free league priors for cold starts
    if league_team_games > 0:
        prior_ppg = league_pts_sum / league_team_games
        prior_gf = league_gf_sum / league_team_games
        prior_ga = prior_gf
        prior_wr = league_win_sum / league_team_games
    else:
        prior_ppg, prior_gf, prior_ga, prior_wr = 1.38, 1.40, 1.40, 0.38

    # 2. Rolling form (last 5 & last 10)
    h5 = compute_window_stats(team_all_history[home], 5, prior_ppg, prior_gf, prior_ga, prior_wr)
    h10 = compute_window_stats(team_all_history[home], 10, prior_ppg, prior_gf, prior_ga, prior_wr)
    a5 = compute_window_stats(team_all_history[away], 5, prior_ppg, prior_gf, prior_ga, prior_wr)
    a10 = compute_window_stats(team_all_history[away], 10, prior_ppg, prior_gf, prior_ga, prior_wr)

    # 3. Venue-specific form (last 5 home for home team, last 5 away for away team)
    h_home5 = compute_window_stats(team_home_history[home], 5, prior_ppg, prior_gf, prior_ga, prior_wr)
    a_away5 = compute_window_stats(team_away_history[away], 5, prior_ppg, prior_gf, prior_ga, prior_wr)

    # 4. Rest days (capped at 14)
    h_rest = min((m_date - team_last_date[home]).days, MAX_REST_DAYS) if home in team_last_date else MAX_REST_DAYS
    a_rest = min((m_date - team_last_date[away]).days, MAX_REST_DAYS) if away in team_last_date else MAX_REST_DAYS

    # 5. Head-to-head history (strictly prior meetings)
    h2h = compute_h2h_stats(h2h_history[(home, away)])

    # 6. Season context
    h_mp = team_season_matches[(season, home)]
    a_mp = team_season_matches[(season, away)]
    gameweek = max(h_mp, a_mp) + 1

    # Assemble feature row
    feature_rows.append({
        "season": season,
        "Match_Date": m_date.strftime("%Y-%m-%d"),
        "Home_Team": home,
        "Away_Team": away,
        "Full_Time_Result": ftr,
        "target": TARGET_MAP[ftr],
        "gameweek": gameweek,
        "home_matches_played": h_mp,
        "away_matches_played": a_mp,
        # Rolling 5
        "home_ppg_5": h5["ppg"], "home_gf_5": h5["gf"], "home_ga_5": h5["ga"], "home_gd_5": h5["gd"], "home_wr_5": h5["wr"],
        "away_ppg_5": a5["ppg"], "away_gf_5": a5["gf"], "away_ga_5": a5["ga"], "away_gd_5": a5["gd"], "away_wr_5": a5["wr"],
        # Rolling 10
        "home_ppg_10": h10["ppg"], "home_gf_10": h10["gf"], "home_ga_10": h10["ga"], "home_gd_10": h10["gd"], "home_wr_10": h10["wr"],
        "away_ppg_10": a10["ppg"], "away_gf_10": a10["gf"], "away_ga_10": a10["ga"], "away_gd_10": a10["gd"], "away_wr_10": a10["wr"],
        # Venue-specific (last 5)
        "home_venue_ppg_5": h_home5["ppg"], "home_venue_gf_5": h_home5["gf"], "home_venue_ga_5": h_home5["ga"],
        "away_venue_ppg_5": a_away5["ppg"], "away_venue_gf_5": a_away5["gf"], "away_venue_ga_5": a_away5["ga"],
        # Rest days
        "home_rest_days": h_rest,
        "away_rest_days": a_rest,
        "rest_days_diff": h_rest - a_rest,
        # Form differentials (helpful for linear & tree models)
        "ppg_diff_5": h5["ppg"] - a5["ppg"],
        "gd_diff_5": h5["gd"] - a5["gd"],
        "ppg_diff_10": h10["ppg"] - a10["ppg"],
        "gd_diff_10": h10["gd"] - a10["gd"],
        # H2H
        "h2h_home_wr": h2h["h2h_home_wr"],
        "h2h_draw_r": h2h["h2h_draw_r"],
        "h2h_away_wr": h2h["h2h_away_wr"],
        "h2h_gd": h2h["h2h_gd"],
    })

    # --- UPDATE STATE AFTER MATCH ---
    h_pts = 3 if ftr == "H" else (1 if ftr == "D" else 0)
    a_pts = 3 if ftr == "A" else (1 if ftr == "D" else 0)

    h_rec = {"pts": h_pts, "gf": hg, "ga": ag}
    a_rec = {"pts": a_pts, "gf": ag, "ga": hg}

    team_all_history[home].append(h_rec)
    team_all_history[away].append(a_rec)
    team_home_history[home].append(h_rec)
    team_away_history[away].append(a_rec)

    team_last_date[home] = m_date
    team_last_date[away] = m_date
    team_season_matches[(season, home)] += 1
    team_season_matches[(season, away)] += 1

    h2h_history[(home, away)].append({"gd": hg - ag})
    h2h_history[(away, home)].append({"gd": ag - hg})

    league_pts_sum += (h_pts + a_pts)
    league_gf_sum += (hg + ag)
    league_win_sum += (1 if h_pts == 3 else 0) + (1 if a_pts == 3 else 0)
    league_team_games += 2

features_df = pd.DataFrame(feature_rows)
print(f"Generated raw feature table: {features_df.shape}")

Generated raw feature table: (2350, 46)


In [4]:
# Drop first 3 gameweeks of 2020-21 (cold start with no prior history)
cold_start_mask = (features_df["season"] == "2020-21") & (features_df["gameweek"] <= 3)
dropped_count = cold_start_mask.sum()
features_df = features_df[~cold_start_mask].reset_index(drop=True)

# Sanity check: zero NaNs in feature matrix
assert features_df.isna().sum().sum() == 0, "NaN values found in features_df!"

features_df.to_csv(FEATURES_FILE, index=False)
print(f"Dropped {dropped_count} cold-start rows (2020-21 GW 1-3).")
print(f"Saved {FEATURES_FILE}: {features_df.shape[0]} rows, {features_df.shape[1]} columns.")
features_df.head()

Dropped 28 cold-start rows (2020-21 GW 1-3).
Saved ..\..\data\processed\features.csv: 2322 rows, 46 columns.


,season,Match_Date,Home_Team,Away_Team,Full_Time_Result,target,gameweek,home_matches_played,away_matches_played,home_ppg_5,...,away_rest_days,rest_days_diff,ppg_diff_5,gd_diff_5,ppg_diff_10,gd_diff_10,h2h_home_wr,h2h_draw_r,h2h_away_wr,h2h_gd
0,2020-21,2020-10-03,Chelsea,Crystal Palace,H,2,4,3,3,1.333333,...,7,0,-0.666667,-0.666667,-0.666667,-0.666667,0.43,0.24,0.33,0.0
1,2020-21,2020-10-03,Everton,Brighton,H,2,4,3,3,3.000000,...,7,0,2.000000,1.666667,2.000000,1.666667,0.43,0.24,0.33,0.0
2,2020-21,2020-10-03,Leeds,Man City,D,1,4,3,2,2.000000,...,6,0,0.500000,0.833333,0.500000,0.833333,0.43,0.24,0.33,0.0
3,2020-21,2020-10-03,Newcastle,Burnley,H,2,4,3,2,1.333333,...,7,-1,1.333333,1.166667,1.333333,1.166667,0.43,0.24,0.33,0.0
4,2020-21,2020-10-04,Arsenal,Sheffield United,H,2,4,3,3,2.000000,...,7,-1,2.000000,2.000000,2.000000,2.000000,0.43,0.24,0.33,0.0


In [5]:
# Compute final league priors
final_ppg = league_pts_sum / league_team_games
final_gf = league_gf_sum / league_team_games
final_wr = league_win_sum / league_team_games

latest_season = df["season"].iloc[-1]
all_teams = sorted(team_last_date.keys())
latest_rows = []

for team in all_teams:
    s5 = compute_window_stats(team_all_history[team], 5, final_ppg, final_gf, final_gf, final_wr)
    s10 = compute_window_stats(team_all_history[team], 10, final_ppg, final_gf, final_gf, final_wr)
    sh5 = compute_window_stats(team_home_history[team], 5, final_ppg, final_gf, final_gf, final_wr)
    sa5 = compute_window_stats(team_away_history[team], 5, final_ppg, final_gf, final_gf, final_wr)

    latest_rows.append({
        "team": team,
        "last_match_date": team_last_date[team].strftime("%Y-%m-%d"),
        "current_season": latest_season,
        "season_matches_played": team_season_matches[(latest_season, team)],
        "ppg_5": s5["ppg"], "gf_5": s5["gf"], "ga_5": s5["ga"], "gd_5": s5["gd"], "wr_5": s5["wr"],
        "ppg_10": s10["ppg"], "gf_10": s10["gf"], "ga_10": s10["ga"], "gd_10": s10["gd"], "wr_10": s10["wr"],
        "venue_home_ppg_5": sh5["ppg"], "venue_home_gf_5": sh5["gf"], "venue_home_ga_5": sh5["ga"],
        "venue_away_ppg_5": sa5["ppg"], "venue_away_gf_5": sa5["gf"], "venue_away_ga_5": sa5["ga"],
    })

latest_state_df = pd.DataFrame(latest_rows)
latest_state_df.to_csv(LATEST_STATE_FILE, index=False)
print(f"Saved {LATEST_STATE_FILE}: {len(latest_state_df)} teams.")
latest_state_df.head()

Saved ..\..\data\processed\team_latest_state.csv: 30 teams.


,team,last_match_date,current_season,season_matches_played,ppg_5,gf_5,ga_5,gd_5,wr_5,ppg_10,gf_10,ga_10,gd_10,wr_10,venue_home_ppg_5,venue_home_gf_5,venue_home_ga_5,venue_away_ppg_5,venue_away_gf_5,venue_away_ga_5
0,Arsenal,2026-10-18,2026-27,7,2.4,1.4,1.0,0.4,0.8,2.7,1.5,0.6,0.9,0.9,3.0,2.2,0.4,2.4,1.2,0.8
1,Aston Villa,2026-10-17,2026-27,7,1.6,1.4,1.2,0.2,0.4,1.5,1.5,1.6,-0.1,0.4,0.8,1.6,1.8,2.0,1.2,1.4
2,Bournemouth,2026-10-18,2026-27,7,1.0,1.4,2.0,-0.6,0.2,1.1,1.2,1.5,-0.3,0.2,1.2,1.2,1.0,1.0,1.2,2.0
3,Brentford,2026-10-17,2026-27,7,1.2,1.8,1.4,0.4,0.2,1.2,1.6,1.4,0.2,0.2,1.6,2.0,1.0,0.8,1.2,1.8
4,Brighton,2026-10-18,2026-27,7,2.6,3.2,0.2,3.0,0.8,1.9,2.6,0.9,1.7,0.6,2.0,2.6,0.8,1.2,2.2,1.6
